# Field factory — add a new queryable content field with review-only human effort

The Bucket-3 pipeline (detect → sample → label → validate → calibrate → query) existed once,
hardcoded for `alienation_alleged`. This notebook is the **parameterised version**: a user
supplies a field name, a one-paragraph definition and a few seed terms — the factory does
the rest, and the human's only task is **reviewing ~120 drafted labels** (agree/flip), not
creating anything.

`FIELD_KIND` in the config cell picks what a cell holds: `"boolean"` (does the case have
property X?) or `"numeric"` (what number does the case state for X?). Everything below is the
same either way — only the drafting prompt, the validation metrics and the deployed column
type follow the kind.

Pipeline (all automated except step 5):
1. **Lexicon expansion** — a local LLM expands the seed terms into surface-form patterns
   (printed; editable in the config cell).
2. **Candidate detection** — sentence-level lexicon hits with section context → transparent
   mention-based confidence per case.
3. **Stratified sample** — by confidence band, same design as the alienation gold set
   (all high/mid bands, sampled negatives).
4. **LLM drafting** — the local LLM applies the definition to each sampled case's evidence
   pool (checkpointed) → `draft_label` / `draft_value` + evidence.
5. **Human review** — fill `gold_<field>` in the template. *The only manual step.* For a
   numeric field write the number, or the word `none` when the case states no value; a blank
   cell means "not reviewed" and is skipped.
6. **Validation** — against the reviewed gold: P/R/F1 for a boolean field; exact-match, ±1,
   MAE and value-presence P/R/F1 for a numeric one. Both then get out-of-fold isotonic
   calibration (ECE) → report + deployable calibrated column.

**Stated honesty caveats, by construction:**
- the drafts ARE the LLM extractor's predictions — so the "LLM F1" measured in step 6 is
  agreement with labels the human reviewed *while seeing those drafts* (anchoring). The
  **flip rate** is therefore reported prominently: a near-zero flip rate means the human
  rubber-stamped, and the validation is weak; a substantial flip rate (the alienation
  reference: 37/120 against the rules extractor) means genuine adjudication.
- domain of validity = the corpus this runs on (ECHR-English here), per field, as always.

## 1. Config — the ONLY cell a user edits
`FIELDS` holds one entry per field (kind, definition, seed terms, question terms) and
`FIELD_ID` picks the one this run works on. Everything downstream reads the derived
`FIELD_NAME` / `FIELD_KIND` / `FIELD_DEFINITION` / `SEED_TERMS`, so adding a field is
adding a dict entry and changing one line.

In [2]:
from pathlib import Path

# ==== the things a user provides: one entry per field, FIELD_ID picks the one to work on ====
# Definitions carry NO worked examples on purpose: a 3B model quotes them back as answers
# (measured: 9 of 40 drafts returned the definition's own example as their evidence).
FIELD_ID = "expert_opinion_ordered"

FIELDS = {
    "child_age": dict(
        kind="numeric", unit="years",
        definition=(
            "child_age = the age in completed years of the child concerned by the family-law "
            "dispute, at the time of the domestic proceedings the case is about. Where several "
            "children are concerned, report the youngest of them. Report a number only when the "
            "case states that child's age, or states a date of birth for that child from which "
            "the age follows; report null when the case states neither. A number is not a "
            "child_age when it is the age of an adult, the age of a child in a different case "
            "quoted as precedent, or a statutory age threshold describing what a court must do "
            "at a given age."),
        seeds=["years old", "months old", "aged", "age of", "date of birth",
               "born on", "born in"],
        question_terms=["child age", "age of the child", "age of the children",
                        "children's age", "age of kids", "how old"]),

    "expert_opinion_ordered": dict(
        kind="boolean",
        definition=(
            "expert_opinion_ordered = TRUE if a court or child-welfare authority in the domestic "
            "proceedings commissioned, ordered or obtained an opinion from a psychological, "
            "psychiatric, medical or child-development expert about the child, a parent or the "
            "family relationship. It stays TRUE whether or not the opinion was delivered, "
            "accepted or followed. It is FALSE if such an opinion was only requested by a party "
            "or discussed as a possibility without being ordered, if the expert evidence belongs "
            "to a different case quoted as precedent, if expert evidence appears only in quoted "
            "legislation, or if the only professional input is a social worker's routine welfare "
            "report rather than an expert opinion."),
        seeds=["expert opinion", "expert report", "psychological report", "psychiatric report",
               "court-appointed expert", "expert evidence", "psychologist", "expert witness"],
        question_terms=["expert opinion", "psychological expert", "expert ordered",
                        "psychologist", "expert evidence"]),

    "child_heard": dict(
        kind="boolean",
        definition=(
            "child_heard = TRUE if the child concerned was heard in the domestic proceedings: "
            "interviewed, questioned or consulted by a judge, a court-appointed expert, a "
            "guardian or a child-welfare authority, in person or through a representative, so "
            "that the child's own views reached the decision-maker. It is FALSE if the child's "
            "views were only relayed second-hand by a parent, if hearing the child was requested "
            "or discussed but did not take place, if the court declined to hear the child, or if "
            "the passage states the right of a child to be heard as a legal standard without "
            "stating that this child was heard."),
        seeds=["heard the child", "the child was heard", "interviewed the child",
               "child was interviewed", "wishes of the child", "views of the child",
               "opinion of the child", "right to be heard"],
        question_terms=["child heard", "was the child heard", "child's views",
                        "hear the child", "children heard"]),

    "coercive_measures": dict(
        kind="boolean",
        definition=(
            "coercive_measures = TRUE if an authority in the domestic proceedings actually "
            "imposed or carried out a coercive measure to enforce contact with, or the return "
            "of, the child: a fine or penalty payment, detention or arrest, removal of the child "
            "by police or a bailiff, or a transfer of custody imposed as a sanction for "
            "obstructing contact. It is FALSE if such a measure was only requested, threatened "
            "in general terms, or described as legally available without being applied, FALSE if "
            "it appears only in quoted legislation or in a different case cited as precedent, "
            "and FALSE where enforcement consisted only of persuasion, counselling or mediation."),
        seeds=["coercive measure", "penalty payment", "fine was imposed", "police assistance",
               "enforcement measures", "forcible removal", "bailiff", "imposed a fine"],
        question_terms=["coercive measures", "enforcement measures", "fines", "penalty payment",
                        "police enforcement"]),

    "applicant_is_father": dict(
        kind="boolean",
        definition=(
            "applicant_is_father = TRUE if the person who brought the case to the Court is the "
            "father of the child concerned by the dispute, including where several applicants "
            "act together and the father is one of them. It is FALSE if the applicant is the "
            "mother, a grandparent, another relative, the child, a foster or adoptive parent, or "
            "an institution, and FALSE if the case concerns no child or does not state the "
            "applicant's relationship to the child."),
        seeds=["is the father", "father of the child", "the applicant's son",
               "the applicant's daughter", "his son", "his daughter", "the applicant, the father"],
        question_terms=["applicant is father", "fathers", "mothers or fathers",
                        "who brings the case", "applicant father"]),
}

_spec = FIELDS[FIELD_ID]
FIELD_NAME      = FIELD_ID
FIELD_KIND      = _spec["kind"]            # "boolean" | "numeric"
FIELD_UNIT      = _spec.get("unit")        # numeric fields only: what the number counts
FIELD_DEFINITION = _spec["definition"]
SEED_TERMS      = _spec["seeds"]
QUESTION_TERMS  = _spec.get("question_terms", [])   # how a question names the deployed field

# ==== knobs (defaults fine) ====
CORPUS_FILE  = Path("../data/echr_parental_alienation.json")   # domain of validity: ECHR-EN
SAMPLE_N     = 120
SEED         = 42
LLM_MODEL    = "llama3.2"
MAX_SENTS    = 10
DATA_DIR     = Path("../data")
REPORT_DIR   = Path("../reports")
TEMPLATE     = DATA_DIR / f"field_{FIELD_NAME}_template.csv"
LABELS       = DATA_DIR / f"field_{FIELD_NAME}_labels.csv"
CKPT         = DATA_DIR / f"field_{FIELD_NAME}_llm_checkpoint.json"

NUMERIC  = FIELD_KIND == "numeric"
CELL_KEY = "value" if NUMERIC else "label"      # what one drafted cell is called throughout
print(f"field: {FIELD_NAME} ({FIELD_KIND}"
      + (f", {FIELD_UNIT}" if NUMERIC else "") + f") | corpus: {CORPUS_FILE.name} "
      f"| sample N={SAMPLE_N} | {len(FIELDS)} fields defined")

field: child_age (numeric, years) | corpus: echr_parental_alienation.json | sample N=120 | 5 fields defined


## 2. Lexicon expansion (LLM, printed for a human glance)
Seed terms → surface-form patterns. The expansion is *suggested*, printed, and the final
list is whatever `LEXICON` ends up containing — edit here if the LLM proposed junk.

In [4]:
import json
import re

OLLAMA_OK = False
try:
    import ollama
    ollama.list()
    OLLAMA_OK = True
except Exception as e:
    print(f"Ollama unreachable ({e}) — using seed terms only")

# The expansion is an LLM call, so the result is FROZEN the first time it runs and reloaded
# afterwards: the detector that drew the gold sample must be the one that later extracts the
# corpus. Without this, a deploy run with ollama down silently fell back to seed terms and
# changed the candidate set (expert_opinion_ordered: 799 candidates when validated, 789 without).
LEXICON_FILE = DATA_DIR / f"field_{FIELD_NAME}_lexicon.json"
LEXICON = list(SEED_TERMS)
if LEXICON_FILE.exists():
    LEXICON = json.loads(LEXICON_FILE.read_text())
    print(f"lexicon loaded from {LEXICON_FILE.name} — frozen with the sample, not re-expanded")
elif OLLAMA_OK:
    prompt = (f"List 10 additional short English surface forms (words or phrases, one per "
              f"line, no numbering, no prose) that ECHR judgments use for this concept:\n"
              f"{FIELD_DEFINITION}\nAlready known: {', '.join(SEED_TERMS)}")
    r = ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}],
                    options={"temperature": 0})
    suggested = [l.strip(" -*\u2022").lower() for l in r["message"]["content"].splitlines()
                 if 2 < len(l.strip()) < 60 and not l.strip().endswith(":")]
    print("LLM-suggested additions (review; junk is harmless — it only adds candidates):")
    for s in suggested:
        print("  +", s)
    LEXICON += suggested

if not LEXICON_FILE.exists():
    LEXICON_FILE.write_text(json.dumps(sorted(set(LEXICON))))

LEX_RE = re.compile("|".join(re.escape(t) for t in sorted(set(LEXICON), key=len, reverse=True)),
                    re.IGNORECASE)
print(f"\nfinal lexicon: {len(set(LEXICON))} patterns")

LLM-suggested additions (review; junk is harmless — it only adds candidates):
  + child_age
  + age of the child
  + youngest child
  + minor
  + adolescent
  + youth
  + young person
  + teenager
  + youngster
  + infant

final lexicon: 17 patterns


## 3. Candidate detection + transparent confidence
Reuses the deployed ECHR section splitter (exec'd from `echr_extraction.ipynb`). Confidence
is a simple, printable function of mention density and section context — it exists to
*stratify the sample*, not to be the final extractor.

In [6]:
from pathlib import Path as _P

src_ex = json.loads(_P("echr_extraction.ipynb").read_text())
_ns = {"re": re, "json": json}
for marker in ["ECHR_ANCHORS = ["]:
    cell = next("".join(c["source"]) for c in src_ex["cells"]
                if c["cell_type"] == "code" and marker in "".join(c["source"]))
    exec(cell, _ns)
echr_sections, _SENT = _ns["echr_sections"], _ns["_SENT"]

records = json.loads(CORPUS_FILE.read_text())
print(f"corpus: {len(records)} records")

def detect(full_text):
    """(confidence, evidence pool) — mentions weighted by section context."""
    secs = echr_sections(full_text or "")
    pool, n_law, n_facts = [], 0, 0
    for lab, t in secs:
        for s in _SENT.split(t):
            if LEX_RE.search(s):
                pool.append((lab, s.strip()[:350]))
                if lab == "LAW":
                    n_law += 1
                elif lab in ("FACTS", "HEADER", "PROCEDURE", "unparsed"):
                    n_facts += 1
    if not pool:
        return 0.02, []
    conf = min(1.0, 0.3 + 0.12 * min(n_facts, 3) + 0.08 * min(n_law, 3))
    return round(conf, 3), pool[:MAX_SENTS]

det = {}
for r in records:
    conf, pool = detect(r.get("full_text", ""))
    det[r["itemid"]] = {"conf": conf, "pool": pool,
                        "title": r.get("docname", ""), "genre": r.get("doctype", ""),
                        "url": f"https://hudoc.echr.coe.int/eng?i={r['itemid']}"}
import numpy as np
confs = np.array([d["conf"] for d in det.values()])
print(f"cases with >=1 mention: {(confs > 0.02).sum()} | conf>=0.5: {(confs >= 0.5).sum()}")

section splitter ready | applicant-context sections: {'HEADER', 'FACTS', 'unparsed', 'PROCEDURE'}
corpus: 1574 records
cases with >=1 mention: 1477 | conf>=0.5: 1260


## 4. Stratified sample → LLM-drafted template (frozen once)

In [8]:
import pandas as pd

if LABELS.exists() or TEMPLATE.exists():
    print("template/labels already exist — frozen, not regenerated")
else:
    rng = np.random.default_rng(SEED)
    ids = list(det)
    conf = {i: det[i]["conf"] for i in ids}
    bands = {
        "high(>=.5)":  [i for i in ids if conf[i] >= 0.5],
        "mid[.3,.5)":  [i for i in ids if 0.3 <= conf[i] < 0.5],
        "none(<.3)":   [i for i in ids if conf[i] < 0.3],
    }
    targets = {"high(>=.5)": 60, "mid[.3,.5)": 40, "none(<.3)": 20}
    picks = []
    for name, pool_ids in bands.items():
        k = min(targets[name], len(pool_ids))
        take = pool_ids if k == len(pool_ids) else list(rng.choice(pool_ids, k, replace=False))
        picks += take
        print(f"  {name:12s}: {len(take)}/{len(pool_ids)}")
    picks = picks[:SAMPLE_N]

    # LLM drafts one cell per sampled case (checkpointed) — these ARE the extractor's predictions
    if NUMERIC:
        PROMPT = ("You extract one number from excerpts of an ECHR family-law case.\n\n"
                  "Definition: " + FIELD_DEFINITION +
                  "\n\nSentences from the case (with document section):\n{sents}\n\n"
                  "Reply ONLY with a JSON object: "
                  '{{"value": <the number, or null if the case does not state one>, '
                  '"confidence": 0.0-1.0, '
                  '"evidence": "<best supporting sentence, verbatim>"}}')
    else:
        PROMPT = ("You classify excerpts from an ECHR family-law case.\n\nDefinition: "
                  + FIELD_DEFINITION +
                  "\n\nSentences from the case (with document section):\n{sents}\n\n"
                  "Reply ONLY with a JSON object: "
                  '{{"label": true or false, "confidence": 0.0-1.0, '
                  '"evidence": "<best supporting sentence, verbatim>"}}')

    def _cell(obj):
        """One drafted cell from the model's JSON — a number, or a flag."""
        if not NUMERIC:
            return bool(obj.get("label"))
        v = obj.get("value")
        return None if v is None or v == "" else float(v)
    _JSON = re.compile(r"\{.*\}", re.S)
    done = json.loads(CKPT.read_text()) if CKPT.exists() else {}
    todo = [i for i in picks if i not in done]
    print(f"LLM drafting: {len(done)} cached, {len(todo)} to run")
    for n, iid in enumerate(todo, 1):
        pool = det[iid]["pool"]
        if not pool or not OLLAMA_OK:
            done[iid] = {CELL_KEY: None if NUMERIC else False, "conf": 0.02, "evidence": None}
        else:
            sents = "\n".join(f"- [{lab}] {s}" for lab, s in pool)
            out = None
            for _ in range(2):
                try:
                    r = ollama.chat(model=LLM_MODEL,
                                    messages=[{"role": "user", "content": PROMPT.format(sents=sents)}],
                                    format="json",   # the parser reads one JSON object, so let the
                                    # model emit only that: without it llama3.2
                                    # generated 429 tokens for an ~80-token
                                    # answer and a call took 3 min instead of 35 s
                                    options={"temperature": 0, "num_predict": 200})
                    m = _JSON.search(r["message"]["content"])
                    obj = json.loads(m.group(0))
                    out = {CELL_KEY: _cell(obj),
                           "conf": max(0.0, min(1.0, float(obj.get("confidence", 0.5)))),
                           "evidence": str(obj.get("evidence", ""))[:300]}
                    break
                except Exception as exn:
                    err = exn
            done[iid] = out or {CELL_KEY: None if NUMERIC else False, "conf": 0.5,
                            "evidence": "unparseable"}
        if n % 10 == 0 or n == len(todo):
            CKPT.write_text(json.dumps(done))
            print(f"  {n}/{len(todo)}")

    draft_col = "draft_value" if NUMERIC else "draft_label"
    rows = [{"id": i, "title": det[i]["title"], "url": det[i]["url"],
             "detector_conf": det[i]["conf"],
             draft_col: done[i][CELL_KEY] if NUMERIC else int(done[i][CELL_KEY]),
             "draft_conf": done[i]["conf"], "draft_evidence": done[i]["evidence"],
             f"gold_{FIELD_NAME}": "", "notes": ""} for i in picks]
    tmpl = pd.DataFrame(rows).sample(frac=1, random_state=SEED)
    tmpl.to_csv(TEMPLATE, index=False)
    if NUMERIC:
        print(f"\nwrote {TEMPLATE.name}: {len(tmpl)} rows | the model proposed a number in "
              f"{int(tmpl.draft_value.notna().sum())} of them")
        print(f"NEXT (the only human step): fill gold_{FIELD_NAME} in every row — the number in "
              f"{FIELD_UNIT} if the case states one (agree with the draft or overrule it), or "
              f"the word none if it states no {FIELD_NAME.replace('_', ' ')}.")
        print("   A blank cell means 'not reviewed' and is skipped, so none is not a blank.")
    else:
        print(f"\nwrote {TEMPLATE.name}: {len(tmpl)} rows | drafted positive: "
              f"{tmpl.draft_label.sum()} ({tmpl.draft_label.mean()*100:.0f}%)")
        print(f"NEXT (the only human step): review gold_{FIELD_NAME} (agree=copy draft, or flip),")
    print(f"save as {LABELS.name}, re-run this notebook.")

template/labels already exist — frozen, not regenerated


## 5. Validation — runs once labels exist (graceful until then)
Reports the **flip rate** first (how often the human overruled the drafts — the integrity
measure), then P/R/F1 of lexicon detector and LLM drafts against gold, then out-of-fold
calibration of the draft confidences.

In [10]:
if not LABELS.exists():
    print(f"no labels yet — review {TEMPLATE.name}, save as {LABELS.name}, re-run.")
else:
    from sklearn.isotonic import IsotonicRegression
    from sklearn.model_selection import StratifiedKFold
    import sys
    if "." not in sys.path:
        sys.path.insert(0, ".")            # notebooks run from src/
    from eval_metrics import ece as _ece   # one definition; n_bins is required, never defaulted
    N_BINS = 5
    ece = lambda conf, yy: _ece(conf, yy, N_BINS)
    report = []

    if NUMERIC:
        # a numeric gold cell is a number, or the word "none" for "the case states no value";
        # a BLANK cell means "not reviewed" and is skipped, so the two are never confused.
        lab = pd.read_csv(LABELS, keep_default_na=False, dtype=str)
        g = lab[f"gold_{FIELD_NAME}"].str.strip().str.lower()
        lab, g = lab[g != ""].copy(), g[g != ""]
        lab["gold"] = [np.nan if v in ("none", "null", "na", "n/a", "-")
                       else pd.to_numeric(v, errors="coerce") for v in g]
        for c in ("draft_value", "draft_conf", "detector_conf"):
            lab[c] = pd.to_numeric(lab[c], errors="coerce")

        both = lab.gold.notna() & lab.draft_value.notna()
        correct = (both & (lab.gold == lab.draft_value)) | (lab.gold.isna() & lab.draft_value.isna())
        flips = int((~correct).sum())
        err = (lab.gold[both] - lab.draft_value[both]).abs()
        tp, fp = int(both.sum()), int((lab.gold.isna() & lab.draft_value.notna()).sum())
        fn = int((lab.gold.notna() & lab.draft_value.isna()).sum())
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        F1 = 2 * P * R / (P + R) if P + R else 0.0
        print(f"labelled: {len(lab)} | gold states a value in {int(lab.gold.notna().sum())} "
              f"cases (median {lab.gold.median():.0f} {FIELD_UNIT})")
        print(f"FLIP RATE (human overruled the draft): {flips}/{len(lab)} = "
              f"{flips/len(lab)*100:.0f}%  <- integrity measure; ~0% = rubber-stamp warning")
        print(f"  LLM drafts: accuracy={correct.mean():.3f} (a cell is right when both say the "
              f"same number, or both say none)")
        print(f"    on the {tp} cases where gold and draft both give a number: "
              f"exact={(err == 0).mean():.3f} within1={(err <= 1).mean():.3f} "
              f"MAE={err.mean():.2f} {FIELD_UNIT}")
        print(f"    does a value exist at all: P={P:.3f} R={R:.3f} F1={F1:.3f}")
        report += [f"- LLM drafts: accuracy {correct.mean():.3f}, exact {(err == 0).mean():.3f}, "
                   f"within ±1 {(err <= 1).mean():.3f}, MAE {err.mean():.2f} {FIELD_UNIT}\n",
                   f"- value-presence: P={P:.3f} R={R:.3f} F1={F1:.3f}\n"]
        headline = (f"accuracy {correct.mean():.3f} / MAE {err.mean():.2f} {FIELD_UNIT} / "
                    f"presence F1 {F1:.3f}")
        # confidence is a claim about a VALUE, so it is calibrated on the cells that have one
        fit = lab.draft_value.notna()
        y, raw = correct[fit].to_numpy().astype(int), lab.draft_conf[fit].to_numpy(float)
        n_reviewed = len(lab)
    else:
        lab = pd.read_csv(LABELS)
        gcol = f"gold_{FIELD_NAME}"
        lab["gold"] = pd.to_numeric(lab[gcol], errors="coerce")
        lab = lab[lab.gold.notna()].copy()
        lab["gold"] = lab.gold.astype(int)
        flips = int((lab.gold != lab.draft_label).sum())
        print(f"labelled: {len(lab)} | gold positive rate {lab.gold.mean():.3f}")
        print(f"FLIP RATE (human overruled the draft): {flips}/{len(lab)} = "
              f"{flips/len(lab)*100:.0f}%  <- integrity measure; ~0% = rubber-stamp warning")
        from sklearn.metrics import precision_recall_fscore_support
        gold = lab.gold.to_numpy()
        for name, pred in [("lexicon detector (conf>=0.5)", (lab.detector_conf >= 0.5).astype(int)),
                           ("LLM drafts", lab.draft_label.astype(int))]:
            p, r, f1, _ = precision_recall_fscore_support(gold, pred, average="binary",
                                                          zero_division=0)
            print(f"  {name:28s}: P={p:.3f} R={r:.3f} F1={f1:.3f}")
            report.append(f"- {name}: P={p:.3f} R={r:.3f} F1={f1:.3f}\n")
        headline = f"F1 {f1:.3f}"
        y, raw = gold, lab.draft_conf.to_numpy(float)
        n_reviewed = len(lab)

    ece_raw, ece_oof = ece(raw, y.astype(float)), float("nan")
    if len(set(y)) > 1 and min((y == 1).sum(), (y == 0).sum()) >= 5:
        oof = np.full_like(raw, np.nan)
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(raw.reshape(-1, 1), y):
            iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
            iso.fit(raw[tr], y[tr])
            oof[te] = iso.predict(raw[te])
        ece_oof = ece(oof, y.astype(float))
        print(f"  ECE raw={ece_raw:.3f} | OOF-calibrated={ece_oof:.3f}  (n_bins={N_BINS})")

    lines = [f"# Field factory report: `{FIELD_NAME}` ({FIELD_KIND})\n\n",
             f"- definition: {FIELD_DEFINITION}\n"] + report + [
             f"- sample: {n_reviewed} reviewed (stratified), headline: {headline}\n",
             f"- **flip rate {flips}/{n_reviewed}** (human vs drafts; low = anchoring warning)\n",
             f"- calibration: ECE raw {ece_raw:.3f} -> OOF {ece_oof:.3f} (n_bins={N_BINS}), "
             f"target = the drafted cell is correct\n",
             f"- protocol: LLM-drafted labels reviewed by a single human annotator while "
             f"seeing the drafts (anchoring caveat, as with all gold sets in this project); "
             f"domain of validity: {CORPUS_FILE.name} only.\n"]
    (REPORT_DIR / f"field_{FIELD_NAME}_report.md").write_text("".join(lines), encoding="utf-8")
    print("wrote", REPORT_DIR / f"field_{FIELD_NAME}_report.md")
    print(f"\nNEXT: field_deploy.ipynb — full-corpus pass + calibrated column + query-layer "
          f"registration (run after the validation is acceptable).")

labelled: 120 | gold states a value in 49 cases (median 9 years)
FLIP RATE (human overruled the draft): 88/120 = 73%  <- integrity measure; ~0% = rubber-stamp warning
  LLM drafts: accuracy=0.267 (a cell is right when both say the same number, or both say none)
    on the 48 cases where gold and draft both give a number: exact=0.271 within1=0.354 MAE=127.95 years
    does a value exist at all: P=0.480 R=0.980 F1=0.644
  ECE raw=0.761 | OOF-calibrated=0.001  (n_bins=5)
wrote ../reports/field_child_age_report.md

NEXT: field_deploy.ipynb — full-corpus pass + calibrated column + query-layer registration (run after the validation is acceptable).
